# DB stuff

A tour of the schema and of the data. Nothing here writes.

It opens `probe.db`, the built database, so it starts in a second instead of
spending two minutes assembling one. To build or rebuild that file:

```bash
rm probe.db && python examples/populate_db.py --lenient
python examples/qc_db.py probe.db          # and check that it merged
```

The next cell also shows how to build one in memory, if you would rather not
depend on the file.

In [1]:
from pathlib import Path

import pandas as pd

from probedb import ProbeDB
from loader import load, load_all
from loader.load import STAGING  # the repository's staging directory

pd.set_option("display.max_colwidth", 44)
pd.set_option("display.width", 170)

DB = STAGING.parent / "probe.db"
if not DB.exists():
    raise FileNotFoundError(
        f"{DB} not found. Build it first:\n"
        f"    python examples/populate_db.py --lenient"
    )

# no create=True: the file already has its tables, and nothing here writes
db = ProbeDB(DB)

# to build from the staging files instead, leaving the file alone:
# db = ProbeDB(":memory:", create=True)
# load_all(db, STAGING, strict=False)            # strict=True refuses a bad directory
# load(db, STAGING / "opnme", source="opnme")    # or one directory at a time

db.counts()

,table,rows
0,compound,16573
1,chembl,15219
2,compound_set,7
3,compound_set_member,20582
4,uniprot,6068
5,target,6908
6,target_uniprot,8584
7,bioactivity_source,51228
8,bioactivity_group,268911
9,bioactivity,457060


## Tables and views

`schema.TABLES` is what is stored, `schema.VIEWS` is what is joined for you.
Both come out of `database/schema.sql`, which is the only definition.

In [2]:
from probedb import schema

columns = []
for table in schema.TABLES + schema.VIEWS:
    for row in db.conn.execute(f"PRAGMA table_info({table})"):
        columns.append({"table": table, "kind": "view" if table in schema.VIEWS else "table",
                        "column": row[1], "type": row[2],
                        "not_null": bool(row[3]), "primary_key": bool(row[5])})

pd.DataFrame(columns)

,table,kind,column,type,not_null,primary_key
0,compound,table,inchikey,VARCHAR(27),False,True
1,compound,table,smiles,TEXT,False,False
2,compound,table,name,VARCHAR(255),False,False
3,chembl,table,chembl_id,VARCHAR(20),False,True
4,chembl,table,inchikey,VARCHAR(27),True,False
...,...,...,...,...,...,...
76,target_flat,view,uniprot_id,VARCHAR(20),False,False
77,target_flat,view,hgnc,VARCHAR(50),False,False
78,target_flat,view,species,VARCHAR(100),False,False
79,target_flat,view,entrez_gene,VARCHAR(50),False,False


In [3]:
db.table("compound")

,inchikey,smiles,name
0,BJFSUDWKXGMUKA-UHFFFAOYSA-N,COc1cc(-c2cn(C)c(=O)c3cnccc23)c(OC)cc1CN...,BI-9564
1,CMJJZRAAQMUAFH-INIZCTEOSA-N,Cc1cn(-c2cc(NC(=O)c3ccc4c(c3)CN(c3cncnc3...,DDR-TRK-1 (D2202-1)
2,GJMZWYLOARVASY-NTCAYCPXSA-N,CN(C)C(=O)/C(C#N)=C/c1ccc(-c2nc3cnc4[nH]...,FM-381
3,RPIGHLXKDWUGDT-DTQAZKPQSA-N,CN(C)C(=O)/C(C#N)=C/c1ccc(-c2nc3cnc4c(cc...,FM-479
4,HEAGNKNMQVIVMM-UHFFFAOYSA-N,CN1C=C(C2=NC3=C(OC=C3C4=CC(C5=CC=NC=C5)=...,MU1210
...,...,...,...
16568,MBNGWHIJMBWFHU-UHFFFAOYSA-N,COc1ccc(-c2cc(=O)c3c(O)cc(O)cc3o2)cc1O,DIOSMETIN
16569,OTKUTGIVEREJLL-UHFFFAOYSA-N,CN1CCN(C2=Cc3ccccc3Oc3ccc(Cl)cc32)CC1,NaN
16570,QISLMXIYRQCLIR-FUMNGEBKSA-N,CSCC[C@H](NC(=O)[C@H](Cc1ccccc1)NC[C@@H]...,NaN
16571,DVEXZJFMOKTQEZ-WHYMJUELSA-N,N#CC(=C(\N)Sc1ccccc1N)/C(C#N)=C(/N)Sc1cc...,NaN


In [4]:
# chembl maps a ChEMBL id to an InChIKey, one row per id
db.table("chembl")

,chembl_id,inchikey
0,CHEMBL1232461,AAAQFGUYHFJNHI-SFHVURJKSA-N
1,CHEMBL364447,AADVCYNFEREWOS-OBRABYBLSA-N
2,CHEMBL1079742,AAKJLRGGTJKAMG-UHFFFAOYSA-N
3,CHEMBL107335,AAOVKJBEBIDNHE-UHFFFAOYSA-N
4,CHEMBL366205,AAOYLOCWJSLLJU-UHFFFAOYSA-N
...,...,...
15214,CHEMBL90568,MBNGWHIJMBWFHU-UHFFFAOYSA-N
15215,CHEMBL90882,OTKUTGIVEREJLL-UHFFFAOYSA-N
15216,CHEMBL91722,QISLMXIYRQCLIR-FUMNGEBKSA-N
15217,CHEMBL95758,DVEXZJFMOKTQEZ-WHYMJUELSA-N


In [5]:
# membership is a link table, so this view is one row per compound and set.
# a compound in three libraries is three rows here
db.table("compound_flat").head(10)

,inchikey,name,smiles,set_id,set_name,category,source_db
0,BJFSUDWKXGMUKA-UHFFFAOYSA-N,BI-9564,COc1cc(-c2cn(C)c(=O)c3cnccc23)c(OC)cc1CN...,1,EUbOPEN,library,EUbOPEN
1,BJFSUDWKXGMUKA-UHFFFAOYSA-N,BI-9564,COc1cc(-c2cn(C)c(=O)c3cnccc23)c(OC)cc1CN...,2,Novartis_MoA,library,Novartis_MoA
2,BJFSUDWKXGMUKA-UHFFFAOYSA-N,BI-9564,COc1cc(-c2cn(C)c(=O)c3cnccc23)c(OC)cc1CN...,3,Probes_n_Drugs,library,Probes_n_Drugs
3,BJFSUDWKXGMUKA-UHFFFAOYSA-N,BI-9564,COc1cc(-c2cn(C)c(=O)c3cnccc23)c(OC)cc1CN...,4,chemicalprobes,library,chemicalprobes
4,BJFSUDWKXGMUKA-UHFFFAOYSA-N,BI-9564,COc1cc(-c2cn(C)c(=O)c3cnccc23)c(OC)cc1CN...,5,opnme,library,opnme
5,CMJJZRAAQMUAFH-INIZCTEOSA-N,DDR-TRK-1 (D2202-1),Cc1cn(-c2cc(NC(=O)c3ccc4c(c3)CN(c3cncnc3...,1,EUbOPEN,library,EUbOPEN
6,CMJJZRAAQMUAFH-INIZCTEOSA-N,DDR-TRK-1 (D2202-1),Cc1cn(-c2cc(NC(=O)c3ccc4c(c3)CN(c3cncnc3...,3,Probes_n_Drugs,library,Probes_n_Drugs
7,CMJJZRAAQMUAFH-INIZCTEOSA-N,DDR-TRK-1 (D2202-1),Cc1cn(-c2cc(NC(=O)c3ccc4c(c3)CN(c3cncnc3...,4,chemicalprobes,library,chemicalprobes
8,GJMZWYLOARVASY-NTCAYCPXSA-N,FM-381,CN(C)C(=O)/C(C#N)=C/c1ccc(-c2nc3cnc4[nH]...,1,EUbOPEN,library,EUbOPEN
9,GJMZWYLOARVASY-NTCAYCPXSA-N,FM-381,CN(C)C(=O)/C(C#N)=C/c1ccc(-c2nc3cnc4[nH]...,3,Probes_n_Drugs,library,Probes_n_Drugs


In [6]:
db.table("uniprot")

,uniprot_id,entrez_gene,hgnc,species,superfamily
0,Q9NPI1,None,NaN,NaN,NaN
1,Q9H8M2,None,NaN,NaN,NaN
2,P04629,None,NaN,NaN,"Protein kinase superfamily, Tyr protein ..."
3,Q16620,None,NaN,NaN,"Protein kinase superfamily, Tyr protein ..."
4,Q16288,None,NaN,NaN,"Protein kinase superfamily, Tyr protein ..."
...,...,...,...,...,...
6063,O60671,None,RAD1,Homo sapiens,NaN
6064,P57771,None,RGS8,Homo sapiens,NaN
6065,O60930,None,RNASEH1,Homo sapiens,NaN
6066,Q12888,None,TP53BP1,Homo sapiens,NaN


In [7]:
db.table("target")

,target_id,type,name
0,1,protein,BRD7@BRD
1,2,protein,BRD9@BRD
2,3,protein,NTRK1
3,4,protein,NTRK2
4,5,protein,NTRK3
...,...,...,...
6903,6904,complex,Somatostatin receptor type 1 / Somatosta...
6904,6905,complex,DNA topoisomerase 2-beta / DNA topoisome...
6905,6906,protein,TP53-binding protein 1
6906,6907,complex,Tubulin beta-2B chain / Tubulin beta-6 c...


In [8]:
db.table("target_flat")

,target_id,type,name,uniprot_id,hgnc,species,entrez_gene,superfamily
0,1,protein,BRD7@BRD,Q9NPI1,NaN,NaN,None,NaN
1,2,protein,BRD9@BRD,Q9H8M2,NaN,NaN,None,NaN
2,3,protein,NTRK1,P04629,NaN,NaN,None,"Protein kinase superfamily, Tyr protein ..."
3,4,protein,NTRK2,Q16620,NaN,NaN,None,"Protein kinase superfamily, Tyr protein ..."
4,5,protein,NTRK3,Q16288,NaN,NaN,None,"Protein kinase superfamily, Tyr protein ..."
...,...,...,...,...,...,...,...,...
8908,6907,complex,Tubulin beta-2B chain / Tubulin beta-6 c...,Q9BQE3,TUBA1C,Homo sapiens (Human),None,Tubulin family
8909,6907,complex,Tubulin beta-2B chain / Tubulin beta-6 c...,Q9BUF5,TUBB6,Homo sapiens (Human),None,Tubulin family
8910,6907,complex,Tubulin beta-2B chain / Tubulin beta-6 c...,Q9BVA1,TUBB2B,Homo sapiens (Human),None,Tubulin family
8911,6907,complex,Tubulin beta-2B chain / Tubulin beta-6 c...,Q9H4B7,TUBB1,Homo sapiens (Human),None,Tubulin family


In [9]:
# the stored table. source_id is an integer: the name of a source lives in
# bioactivity_source, once, not on every one of its measurements
db.table("bioactivity").head()

,id,inchikey,target_id,moa,bioactivity_type,relation,value,unit,assay_type,assay_description,cell_line,concentration,concentration_unit,source_id,source_xref
0,1,BJFSUDWKXGMUKA-UHFFFAOYSA-N,1,Inhibitor,Kd,=,73.0,nM,biochemical,Competition binding assay (DiscoverX),NaN,1.0,uM,1,https://doi.org/10.1021/acs.jmedchem.5b0...
1,2,BJFSUDWKXGMUKA-UHFFFAOYSA-N,2,Inhibitor,Kd,=,5.9,nM,biochemical,Competition binding assay (DiscoverX),NaN,1.0,uM,1,https://doi.org/10.1021/acs.jmedchem.5b0...
2,3,CMJJZRAAQMUAFH-INIZCTEOSA-N,3,Inhibitor,IC50,=,43.0,nM,biochemical,Enzymatic inhibition assay,NaN,1.0,uM,1,https://www.thesgc.org/chemical-probes/D...
3,4,CMJJZRAAQMUAFH-INIZCTEOSA-N,4,Inhibitor,IC50,=,3.6,nM,biochemical,Enzymatic inhibition assay,NaN,1.0,uM,1,https://www.thesgc.org/chemical-probes/D...
4,5,CMJJZRAAQMUAFH-INIZCTEOSA-N,5,Inhibitor,IC50,=,2.9,nM,biochemical,Enzymatic inhibition assay,NaN,1.0,uM,1,https://www.thesgc.org/chemical-probes/D...


In [10]:
# the same rows with the compound, the target and the source spelled out.
# db.bioactivities() is this view with a WHERE on it
db.table("bioactivity_flat").head()

,id,inchikey,compound,target_id,target_type,target,moa,bioactivity_type,relation,value,...,assay_type,assay_description,cell_line,concentration,concentration_unit,source_id,source_db,source,source_xref,source_url
0,1,BJFSUDWKXGMUKA-UHFFFAOYSA-N,BI-9564,1,protein,BRD7@BRD,Inhibitor,Kd,=,73.0,...,biochemical,Competition binding assay (DiscoverX),NaN,1.0,uM,1,EUbOPEN,NaN,https://doi.org/10.1021/acs.jmedchem.5b0...,NaN
1,2,BJFSUDWKXGMUKA-UHFFFAOYSA-N,BI-9564,2,protein,BRD9@BRD,Inhibitor,Kd,=,5.9,...,biochemical,Competition binding assay (DiscoverX),NaN,1.0,uM,1,EUbOPEN,NaN,https://doi.org/10.1021/acs.jmedchem.5b0...,NaN
2,3,CMJJZRAAQMUAFH-INIZCTEOSA-N,DDR-TRK-1 (D2202-1),3,protein,NTRK1,Inhibitor,IC50,=,43.0,...,biochemical,Enzymatic inhibition assay,NaN,1.0,uM,1,EUbOPEN,NaN,https://www.thesgc.org/chemical-probes/D...,NaN
3,4,CMJJZRAAQMUAFH-INIZCTEOSA-N,DDR-TRK-1 (D2202-1),4,protein,NTRK2,Inhibitor,IC50,=,3.6,...,biochemical,Enzymatic inhibition assay,NaN,1.0,uM,1,EUbOPEN,NaN,https://www.thesgc.org/chemical-probes/D...,NaN
4,5,CMJJZRAAQMUAFH-INIZCTEOSA-N,DDR-TRK-1 (D2202-1),5,protein,NTRK3,Inhibitor,IC50,=,2.9,...,biochemical,Enzymatic inhibition assay,NaN,1.0,uM,1,EUbOPEN,NaN,https://www.thesgc.org/chemical-probes/D...,NaN


## Where a number came from

`source_db` is the resource, `source` is the record inside it, `source_xref`
identifies the one measurement, and `xref_id` is the prefix that turns it into
a link. Only the last three live on `bioactivity_source`, so counting by source
means reading the view.

In [11]:
db.read("SELECT source_db, COUNT(*) n FROM bioactivity_flat GROUP BY 1 ORDER BY n DESC")

,source_db,n
0,ChEMBL,439988
1,Probes & Drugs,6224
2,SPARK-PNAS,5772
3,Chemical Probes Portal,3593
4,EUbOPEN,1408
5,opnMe,75


In [12]:
# every source with how much it contributed, and how to resolve its xref
db.sources()

,source_id,source_db,source,xref_id,measurements
0,4569,ChEMBL,ChEMBL 37,https://doi.org/,92519
1,5003,ChEMBL,PMID:37468498,https://doi.org/,87987
2,4567,ChEMBL,PMID:29191878,https://doi.org/,63937
3,9064,ChEMBL,PMID:22037378,https://doi.org/,21859
4,10663,ChEMBL,PMID:23398362,https://doi.org/,10147
...,...,...,...,...,...
51223,47970,ChEMBL,PMID:9986721,https://doi.org/,0
51224,23553,ChEMBL,PMID:9990446,https://doi.org/,0
51225,22980,ChEMBL,PMID:9990449,https://doi.org/,0
51226,38086,ChEMBL,PMID:9990455,https://doi.org/,0


In [13]:
# where the source has an xref_id, the api builds the link for you
activity = db.bioactivities()

activity[activity.source_url.notna()][
    ["compound", "target", "value", "unit", "source_db", "source", "source_url"]]

,compound,target,value,unit,source_db,source,source_url
1408,NaN,NON-PROTEIN TARGET,2.6,nM,ChEMBL,CHEMBL1145824,211141CHEMBL708673
1409,NaN,PC-3,3.0,nM,ChEMBL,CHEMBL1145824,211141CHEMBL762702
1410,NaN,Unchecked,32.0,%,ChEMBL,CHEMBL1145824,211141CHEMBL844553
1411,Tarceva,Epidermal growth factor receptor,420.0,nM,ChEMBL,CHEMBL1154343,3222537CHEMBL1116652
1412,Tarceva,Epidermal growth factor receptor,15.0,nM,ChEMBL,CHEMBL1154343,3222537CHEMBL1116653
...,...,...,...,...,...,...,...
451283,Bestcall,HTR2A,30000.0,nM,ChEMBL,PMID:37468498,https://doi.org/10.1038/s41467-023-40064-9
451284,Bestcall,HTR2B,30000.0,nM,ChEMBL,PMID:37468498,https://doi.org/10.1038/s41467-023-40064-9
451285,Bestcall,CNR1,30000.0,nM,ChEMBL,PMID:37468498,https://doi.org/10.1038/s41467-023-40064-9
451286,Bestcall,Androgen receptor,30000.0,nM,ChEMBL,PMID:37468498,https://doi.org/10.1038/s41467-023-40064-9


## Complexes

`target` and `uniprot` have no column in common; the link between them is
`target_uniprot`. `db.targets()` does both joins, which is the `target_flat`
view. One row per target and accession, so a complex repeats its `target_id`,
once per subunit.

In [14]:
# an accession belongs to as many targets as contain it: P09874 is PARP1 on its
# own and a member of PARP 1, 2 and 3
db.targets("P09874")

,target_id,type,name,uniprot_id,hgnc,species,entrez_gene,superfamily
0,1549,protein,PARP1,P09874,PARP1,Homo sapiens (Human),None,ARTD/PARP family
1,4252,family,"PARP 1, 2 and 3",P09874,PARP1,Homo sapiens (Human),None,ARTD/PARP family
2,4252,family,"PARP 1, 2 and 3",Q9UGN5,PARP2,Homo sapiens (Human),None,ARTD/PARP family
3,4252,family,"PARP 1, 2 and 3",Q9Y6F1,PARP3,Homo sapiens (Human),None,ARTD/PARP family


## Protein families

`uniprot.superfamily` is UniProt's own classification of a protein, filled in
from `reference/uniprot_protein_families.tsv` so every source agrees on it. It
holds the whole hierarchy, outermost group first, which means one string
answers at several levels.

This is a property of a *protein*. A target of type `family` is a different
thing: our own curated grouping, like `PARP 1, 2 and 3` above.

In [15]:
# db.families() splits the hierarchy into levels and counts what sits under
# each, so you can ask for a superfamily or one family inside it
db.families(like="kinase").head(8)

,family,proteins,targets
0,Protein kinase superfamily,249,474
1,Tyr protein kinase family,58,120
2,AGC Ser/Thr protein kinase family,37,61
3,CAMK Ser/Thr protein kinase family,34,51
4,Ser/Thr protein kinase family,34,52
5,CMGC Ser/Thr protein kinase family,33,107
6,STE Ser/Thr protein kinase family,26,34
7,TKL Ser/Thr protein kinase family,20,35


In [16]:
# "for the RAS family, what is available?". family= works on targets(),
# compounds() and bioactivities()
db.compounds(family="Ras family").head(8)[["name", "sets", "n_targets"]]

,name,sets,n_targets
0,vemurafenib,"Novartis_MoA, reFRAME",372
1,Finlee,"reFRAME, spark",321
2,DP-4978,reFRAME,31
3,Adagrasib,"Probes_n_Drugs, chemicalprobes, reFRAME",16
4,LONAFARNIB,"Probes_n_Drugs, reFRAME, spark",13
5,Iberdomide,"Probes_n_Drugs, chemicalprobes, reFRAME",12
6,L-778123,"Probes_n_Drugs, reFRAME",8
7,opnurasib,reFRAME,7


In [17]:
# before quoting that number, look at what it counted. a family answer is only
# as good as the target rows underneath it, and a `protein` target holding 29
# accessions drags all 29 families in with it. these are PROTAC screens filed
# with the E3 ligase as the target and every neo-substrate as a member
db.read("""
  SELECT t.target_id, t.type, t.name, COUNT(*) AS accessions
    FROM target t JOIN target_uniprot tu ON tu.target_id = t.target_id
   WHERE t.target_id IN (
         SELECT tu2.target_id FROM target_uniprot tu2
           JOIN uniprot u ON u.uniprot_id = tu2.uniprot_id
          WHERE (', ' || u.superfamily || ', ') LIKE '%, Ras family, %')
   GROUP BY t.target_id ORDER BY accessions DESC
""")

,target_id,type,name,accessions
0,2533,protein,Protein cereblon,29
1,2657,protein,GTPase KRas,3
2,6716,complex,SOS1-KRAS,2
3,6717,ppi,von Hippel-Lindau disease tumor suppress...,2
4,1320,protein,KRAS,1
5,1507,protein,NRAS,1


In [18]:
# so the trustworthy version excludes the protein targets holding more than one
# accession. a complex with several members is fine, that is what a complex is
FAMILY = "Ras family"

members = db.targets(family=FAMILY)
per_target = members.groupby(["target_id", "type"], as_index=False).uniprot_id.nunique()
excluded = per_target[(per_target.type == "protein") & (per_target.uniprot_id > 1)].target_id

rows = db.bioactivities(family=FAMILY)
rows = rows[~rows.target_id.isin(excluded)]

hits = (
    rows[rows.unit.eq("nM") & rows.relation.eq("=")]
    .groupby(["inchikey", "compound", "target"], as_index=False)
    .agg(n=("value", "size"), best=("value", "min"))
    .merge(db.compounds()[["inchikey", "sets"]], on="inchikey")
    .sort_values("best")
)

hits.head(12)[["compound", "target", "n", "best", "sets"]]

,compound,target,n,best,sets
7,L-778123,NRAS,1,0.30,"Probes_n_Drugs, reFRAME"
4,ACBI3,KRAS,1,2.00,"Probes_n_Drugs, chemicalprobes, opnme"
11,Adagrasib,KRAS,9,5.00,"Probes_n_Drugs, chemicalprobes, reFRAME"
2,BI-0474,KRAS,1,7.00,"Probes_n_Drugs, opnme"
10,BMS-214662,KRAS,4,8.40,reFRAME
8,sotorasib,KRAS,7,9.66,"Probes_n_Drugs, chemicalprobes, reFRAME"
0,opnurasib,KRAS,6,10.00,reFRAME
9,sotorasib,SOS1-KRAS,1,15.80,"Probes_n_Drugs, chemicalprobes, reFRAME"
3,LONAFARNIB,KRAS,1,40.00,"Probes_n_Drugs, reFRAME, spark"
12,Adagrasib,SOS1-KRAS,1,130.00,"Probes_n_Drugs, chemicalprobes, reFRAME"


In [19]:
# which families are best covered. one compound can reach several levels of one
# hierarchy, so the counting has to be on distinct pairs and not on sums
pairs = db.read("""
  SELECT DISTINCT u.superfamily, g.inchikey
    FROM uniprot u
    JOIN target_uniprot tu ON tu.uniprot_id = u.uniprot_id
    JOIN bioactivity_group g ON g.target_id = tu.target_id
   WHERE u.superfamily IS NOT NULL AND u.superfamily != ''
""")

coverage = (
    pairs.assign(family=pairs.superfamily.str.split(", "))
    .explode("family")
    .groupby("family", as_index=False)
    .inchikey.nunique()
    .rename(columns={"inchikey": "compounds"})
    .merge(db.families(), on="family")
    .sort_values("compounds", ascending=False)
)

coverage.head(10)

,family,compounds,proteins,targets
617,Protein kinase superfamily,3513,249,474
255,G protein-coupled receptor 1 family,3423,190,330
350,Histone deacetylase family,3376,11,49
330,HD type 2 subfamily,3203,6,17
507,Nuclear hormone receptor family,2553,33,73
771,Tyr protein kinase family,2225,58,120
605,Potassium channel family,2063,24,67
327,H (Eag) (TC 1.A.1.20) subfamily,2043,5,11
402,Kv11.1/KCNH2 sub-subfamily,2041,1,2
482,NR3 subfamily,1989,8,16


In [20]:
# the classification also rides along on target_flat, so a target profile can
# be read by family without a second query. this is the selectivity question:
# how far outside its own family does a compound reach
families_of = db.targets()[["target_id", "superfamily"]].dropna().drop_duplicates()

(db.bioactivities(compound="BI-2536")
   .merge(families_of, on="target_id")
   .query("unit == 'nM' and relation == '='")
   .assign(family=lambda d: d.superfamily.str.split(", ").str[0])
   .groupby("family", as_index=False)
   .agg(targets=("target_id", "nunique"), n=("value", "size"), best=("value", "min"))
   .sort_values("best"))

,family,targets,n,best
6,Protein kinase superfamily,52,145,0.083
0,BET family,2,38,1.200
7,TAF1 family,1,2,160.000
1,Carbohydrate kinase PfkB family,1,1,1163.000
4,PI3/PI4-kinase family,2,2,2407.000
5,Potassium channel family,1,2,2900.000
2,Cytochrome P450 family,3,3,3600.000
3,PBTF family,1,1,37000.000


## Sets, or where a compound came from

One staging directory is one compound set, named after the directory. Nobody
writes a file for it: loading `staging/reFRAME` records a set called `reFRAME`
and puts every compound it declared in it.

Membership is its own table rather than a column on the compound, because a
compound arriving from three directories is one compound row and three
memberships. `db.compounds()` rolls that back up to one row per compound.

In [21]:
db.sets()

,set_id,name,category,source_db,compounds
0,6,reFRAME,library,reFRAME,8513
1,3,Probes_n_Drugs,library,Probes_n_Drugs,4840
2,2,Novartis_MoA,library,Novartis_MoA,4185
3,4,chemicalprobes,library,chemicalprobes,1223
4,7,spark,library,spark,987
5,1,EUbOPEN,library,EUbOPEN,732
6,5,opnme,library,opnme,102


In [22]:
# one row per compound. filtering keeps that shape, so a compound in five
# libraries is still one row and still shows all five
compounds = db.compounds()

compounds[compounds.n_sets > 1].head(10)[["name", "n_sets", "sets", "n_targets"]]

,name,n_sets,sets,n_targets
0,molibresib,3,"Novartis_MoA, reFRAME, spark",725
1,Palbociclib,6,"EUbOPEN, Novartis_MoA, Probes_n_Drugs, c...",629
2,Sunitinib,4,"Novartis_MoA, chemicalprobes, reFRAME, s...",611
3,SORAFENIB,4,"Probes_n_Drugs, chemicalprobes, reFRAME,...",588
4,Tasigna,3,"Novartis_MoA, reFRAME, spark",580
5,Tarceva,2,"Novartis_MoA, reFRAME",573
6,Crizotinib,3,"chemicalprobes, reFRAME, spark",571
7,GEFITINIB,4,"Probes_n_Drugs, chemicalprobes, reFRAME,...",571
8,Bosutinib,3,"Novartis_MoA, reFRAME, spark",570
9,roscovitine,3,"Novartis_MoA, reFRAME, spark",563


In [23]:
biggest = db.sets().name.iloc[0]

db.compounds(set=biggest).head()          # everything that came in from one directory

,inchikey,name,smiles,n_sets,sets,n_targets
0,AAAQFGUYHFJNHI-SFHVURJKSA-N,molibresib,CCNC(=O)C[C@@H]1N=C(c2ccc(Cl)cc2)c2cc(OC...,3,"Novartis_MoA, reFRAME, spark",725
1,AHJRHEGDXFFMBM-UHFFFAOYSA-N,Palbociclib,CC(=O)c1c(C)c2cnc(Nc3ccc(N4CCNCC4)cn3)nc...,6,"EUbOPEN, Novartis_MoA, Probes_n_Drugs, c...",629
2,WINHZLLDWRZWRT-ATVHPVEESA-N,Sunitinib,CCN(CC)CCNC(=O)c1c(C)[nH]c(/C=C2\C(=O)Nc...,4,"Novartis_MoA, chemicalprobes, reFRAME, s...",611
3,MLDQJTXFUGDVEO-UHFFFAOYSA-N,SORAFENIB,CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(C(F...,4,"Probes_n_Drugs, chemicalprobes, reFRAME,...",588
4,HHZIURLSWUIHRB-UHFFFAOYSA-N,Tasigna,Cc1cn(-c2cc(NC(=O)c3ccc(C)c(Nc4nccc(-c5c...,3,"Novartis_MoA, reFRAME, spark",580


In [24]:
# the other direction: given a compound, which collections claim it
db.compound_sets(compounds.inchikey.iloc[0])

,set_id,name,category,source_db,description
0,2,Novartis_MoA,library,Novartis_MoA,None
1,6,reFRAME,library,reFRAME,None
2,7,spark,library,spark,None


In [25]:
# a compound is in as many sets as claim it, so the overlap between two
# collections is a join and not a manual comparison
db.read("""
  SELECT a.name AS set_a, b.name AS set_b, COUNT(*) AS shared
    FROM compound_set_member ma JOIN compound_set a ON a.set_id = ma.set_id
    JOIN compound_set_member mb ON mb.inchikey = ma.inchikey
    JOIN compound_set b ON b.set_id = mb.set_id
   WHERE a.set_id < b.set_id
   GROUP BY a.set_id, b.set_id ORDER BY shared DESC
""")

,set_a,set_b,shared
0,Novartis_MoA,reFRAME,1252
1,Probes_n_Drugs,chemicalprobes,833
2,reFRAME,spark,489
3,Novartis_MoA,Probes_n_Drugs,386
4,Probes_n_Drugs,reFRAME,367
5,Novartis_MoA,spark,358
6,chemicalprobes,reFRAME,326
7,EUbOPEN,Probes_n_Drugs,304
8,Novartis_MoA,chemicalprobes,295
9,EUbOPEN,chemicalprobes,260


## One compound

Compounds are searchable by name as well as by InChIKey.

In [26]:
# accepts a name, an InChIKey or a ChEMBL id
db.bioactivities(compound="BI-2536")[
    ["target", "moa", "bioactivity_type", "relation", "value", "unit",
     "assay_type", "cell_line", "source_db", "source"]]

,target,moa,bioactivity_type,relation,value,unit,assay_type,cell_line,source_db,source
0,PLK1,Inhibitor,Kd,=,0.190000,nM,biochemical,NaN,EUbOPEN,NaN
1,BRD4,Inhibitor,Kd,=,37.000000,nM,biochemical,NaN,EUbOPEN,NaN
2,BRD4,Inhibitor,IC50,=,300.000000,nM,cell,NaN,EUbOPEN,NaN
3,PLK3,inhibitor,NaN,NaN,8.050000,,biochemical,NaN,Probes & Drugs,NaN
4,PLK1,inhibitor,NaN,NaN,8.800000,,biochemical,NaN,Probes & Drugs,NaN
...,...,...,...,...,...,...,...,...,...,...
967,PLK1,Serine/threonine-protein kinase PLK1 inh...,IC50,=,12.000000,nM,binding,NaN,ChEMBL,PMID:39965158
968,PLK2,,IC50,=,50.000000,nM,binding,NaN,ChEMBL,PMID:39965158
969,PLK3,,IC50,=,43.000000,nM,binding,NaN,ChEMBL,PMID:39965158
970,BRD4,,pActMean,NaN,7.426667,-log10(M),NaN,NaN,SPARK-PNAS,NaN


## Units

Units are kept as the source reported them, nothing is converted. Probes & Drugs
holds no raw concentrations at all - all 982K of its measured rows are pre-logged, so its rows arrive as pIC50 on
-log(M).

So a comparison has to say which unit it is comparing on.

In [27]:
activity = db.bioactivities()

activity.groupby(["bioactivity_type", "unit"], as_index=False).size()

,bioactivity_type,unit,size
0,% Control,%,2587
1,% Ctrl,%,41
2,% activation,%,1
3,% activity remaining,%,23
4,% inhibition,%,18
...,...,...,...
1307,residual activity,%,1
1308,x fold,uM,2
1309,ΔTm,,1
1310,ΔTm,degC,11


## Repeated measurements

Nothing is averaged on load, so the spread is still there to look at.

In [28]:
groups = (
    activity
    .groupby(["compound", "target", "bioactivity_type", "unit"], as_index=False)
    .agg(n=("value", "size"), minimum=("value", "min"), maximum=("value", "max"),
         median=("value", "median"))
    .sort_values("n", ascending=False)
)

groups.head(10)

,compound,target,bioactivity_type,unit,n,minimum,maximum,median
18475,Acetazolamide,CA2,Ki,nM,509,0.3434,74000.0,12.000
18427,Acetazolamide,CA1,Ki,nM,488,0.7472,36200.0,250.000
225097,Vorinostat,HDAC1,IC50,nM,411,0.2300,33000.0,49.000
18533,Acetazolamide,CA9,Ki,nM,346,0.0600,441.8,25.000
225139,Vorinostat,HDAC6,IC50,nM,341,1.0000,33000.0,32.000
18442,Acetazolamide,CA12,Ki,nM,273,2.5000,339.7,5.700
250914,dosimertinib,EGFR,IC50,nM,253,0.0200,100000.0,12.000
102465,GEFITINIB,EGFR,IC50,nM,246,0.1000,22800.0,22.750
225114,Vorinostat,HDAC2,IC50,nM,242,0.1790,96000.0,120.875
292163,tacrine hydrochloride,Acetylcholinesterase,IC50,nM,242,1.8700,76500.0,109.000


## A worked question, end to end

Everything above looks at one table at a time. This is what actually asking the
database a question looks like: pick a target, find out what is available
against it, where those compounds came from and which of them is the most
potent. Then the same in reverse for one compound.

Every call returns a DataFrame, so the last step is always ordinary pandas.

In [29]:
# which targets are worth asking about at all
covered = db.read("""
  SELECT t.target_id, t.type, t.name, u.hgnc,
         COUNT(DISTINCT b.inchikey) AS compounds, COUNT(*) AS measurements
    FROM bioactivity b
    JOIN target t ON t.target_id = b.target_id
    LEFT JOIN target_uniprot tu ON tu.target_id = t.target_id
    LEFT JOIN uniprot u ON u.uniprot_id = tu.uniprot_id
   GROUP BY t.target_id
   ORDER BY compounds DESC
""")

covered.head(10)

,target_id,type,name,hgnc,compounds,measurements
0,1133,protein,HDAC6,HDAC6,3115,6043
1,2892,protein,Replicase polyprotein 1ab,rep,3096,4588
2,1258,protein,KCNH2,KCNH2,2040,3271
3,213,protein,ADRA2A,NaN,1802,3113
4,202,protein,DRD1,NaN,1785,2767
5,246,protein,HTR1A,NaN,1773,2617
6,226,protein,OPRM1,NaN,1753,2568
7,1937,protein,SLC6A3,SLC6A3,1737,2210
8,231,protein,HTR2B,NaN,1736,2578
9,1938,protein,SLC6A4,SLC6A4,1735,2300


In [30]:
# take the best covered one. TARGET can be an accession, an HGNC symbol or a
# target id, and an accession that belongs to a protein and to the complexes
# containing it returns all of them
TARGET = int(covered["target_id"].iloc[0])

against = db.bioactivities(target=TARGET)
print(f"{covered['name'].iloc[0]}: {len(against)} measurements, "
      f"{against.inchikey.nunique()} compounds")

against[["compound", "moa", "bioactivity_type", "relation", "value", "unit",
         "assay_type", "source_db"]].head()

HDAC6: 6043 measurements, 3115 compounds


,compound,moa,bioactivity_type,relation,value,unit,assay_type,source_db
0,AVS-100,inhibitor,NaN,NaN,12.0,nM,biochemical,Chemical Probes Portal
1,AVS-100,inhibitor,IC50,NaN,500.0,nM,cell,Chemical Probes Portal
2,ACY-738,inhibitor,NaN,NaN,1.7,nM,biochemical,Chemical Probes Portal
3,ACY-738,inhibitor,NaN,NaN,NaN,,cell,Chemical Probes Portal
4,SGC-UBD253,inhibitor,NaN,NaN,84.0,nM,binding,Chemical Probes Portal


In [31]:
# which sets those compounds are in. db.compounds() is one row per compound,
# so this is a plain merge on the InChIKey
available = (
    against[["inchikey", "compound"]].drop_duplicates()
    .merge(db.compounds()[["inchikey", "sets", "n_sets"]], on="inchikey")
    .sort_values("n_sets", ascending=False)
)

available.head(10)

,inchikey,compound,sets,n_sets
133,XQVVPGYIWAGRNI-JOCHJYFZSA-N,BI-2536,"EUbOPEN, Novartis_MoA, Probes_n_Drugs, c...",7
1594,BCFGMOOMADDAQU-UHFFFAOYSA-N,Lapatinib,"EUbOPEN, Novartis_MoA, Probes_n_Drugs, c...",6
669,MUOKSQABCJCOPU-UHFFFAOYSA-N,CX-4945,"EUbOPEN, Novartis_MoA, Probes_n_Drugs, c...",6
3057,KDFQABSFVYLGPM-QFIPXVFZSA-N,L-365260,"EUbOPEN, Novartis_MoA, Probes_n_Drugs, c...",6
160,AQGNHMOJWBZFQQ-UHFFFAOYSA-N,CHIR99021,"EUbOPEN, Novartis_MoA, Probes_n_Drugs, c...",6
210,MVCOAUNKQVWQHZ-UHFFFAOYSA-N,DORAMAPIMOD,"Novartis_MoA, Probes_n_Drugs, chemicalpr...",6
1364,AHJRHEGDXFFMBM-UHFFFAOYSA-N,Palbociclib,"EUbOPEN, Novartis_MoA, Probes_n_Drugs, c...",6
728,STUWGJZDJHPWGZ-LBPRGKRZSA-N,Alpelisib,"EUbOPEN, Novartis_MoA, Probes_n_Drugs, c...",6
918,LIRYPHYGHXZJBZ-UHFFFAOYSA-N,Trametinib,"EUbOPEN, Novartis_MoA, Probes_n_Drugs, c...",6
893,IFSDAJWBUCMOAH-HNNXBMFYSA-N,CAL-101,"EUbOPEN, Novartis_MoA, Probes_n_Drugs, c...",6


In [32]:
# most potent. nothing is converted in the database, so the comparison has to
# name the scale it is comparing on, and a censored value is not a number you
# can rank: ">10000 nM" only says the compound is inactive
potency = against[
    against.unit.eq("nM")
    & against.relation.eq("=")
    & against.bioactivity_type.isin(["IC50", "EC50", "Kd", "Ki"])
]

(potency.groupby(["compound", "bioactivity_type"], as_index=False)
        .agg(n=("value", "size"), best=("value", "min"), median=("value", "median"))
        .sort_values("best")
        .head(10))

,compound,bioactivity_type,n,best,median
110,trichostatin A,Ki,7,0.13,0.990
109,trichostatin A,IC50,92,0.40,4.845
64,Panobinostat,Ki,5,0.70,1.500
63,Panobinostat,IC50,20,1.00,10.025
83,Ricolinostat,Ki,1,1.00,1.000
94,Vorinostat,IC50,341,1.00,32.000
96,Vorinostat,Ki,11,1.00,21.000
16,Belinostat,Ki,3,1.60,1.600
32,Dactolisib,IC50,1,2.00,2.000
6,Acy-241,IC50,17,2.21,2.600


In [33]:
# and the same question from the compound's side: what else does it hit, where
# did it come from, and who says so
COMPOUND = available["inchikey"].iloc[0]

print(db.compound_sets(COMPOUND).to_string(index=False))

profile = db.bioactivities(compound=COMPOUND)
print(f"{profile.compound.iloc[0]}: {len(profile)} measurements over "
      f"{profile.target_id.nunique()} targets, "
      f"{profile.unit.nunique()} different units")

# the same rule as above: rank on one scale or not at all. sorting the whole
# profile by value would put a -5% inhibition readout above a 0.08 nM IC50
(profile[profile.unit.eq("nM") & profile.relation.eq("=")]
    .groupby(["target", "target_type", "bioactivity_type"], as_index=False)
    .agg(n=("value", "size"), best=("value", "min"),
         sources=("source_db", lambda s: ", ".join(sorted(set(s)))))
    .sort_values("best")
    .head(10))

 set_id           name category      source_db description
      1        EUbOPEN  library        EUbOPEN        None
      2   Novartis_MoA  library   Novartis_MoA        None
      3 Probes_n_Drugs  library Probes_n_Drugs        None
      4 chemicalprobes  library chemicalprobes        None
      5          opnme  library          opnme        None
      6        reFRAME  library        reFRAME        None
      7          spark  library          spark        None
BI-2536: 972 measurements over 510 targets, 7 different units


,target,target_type,bioactivity_type,n,best,sources
72,PLK1,protein,IC50,37,0.083,"ChEMBL, opnMe"
73,PLK1,protein,Kd,4,0.190,"ChEMBL, EUbOPEN"
74,PLK1,protein,Ki,2,0.220,ChEMBL
76,PLK2,protein,Kd,1,0.810,ChEMBL
7,BRD4,protein,IC50,18,1.200,"ChEMBL, EUbOPEN"
77,PLK3,protein,IC50,10,1.610,ChEMBL
71,PLK1,protein,EC50,1,2.000,ChEMBL
6,BRD4,protein,EC50,1,2.000,ChEMBL
75,PLK2,protein,IC50,11,2.920,ChEMBL
78,PLK3,protein,Kd,1,4.000,ChEMBL
